# 📓 Notebook 4: Assembly & Output

**Purpose**: Stitch simplified chunks back into a coherent Markdown book, add a table of contents, run quality checks, and export.

**Pipeline**:
```
simplified_chunks.json → Overlap trimming → Reassembly → TOC generation → Quality review → Save .md
```

**Input**: `simplified_chunks.json`  
**Output**: `data/output/simplified_book.md` — the final readable book!

In [ ]:
# ── Imports & Configuration ─────────────────────────────────────────────────
import sys
import json
from pathlib import Path
from IPython.display import display, Markdown

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    SIMPLIFIED_CHUNKS_FILE, BOOK_STRUCTURE_FILE, OUTPUT_DIR, OVERLAP_TOKENS,
)
from src.assembler import (
    trim_overlaps, assemble_book, generate_toc,
    save_output, compare_lengths,
)

In [ ]:
# ── Load Data ────────────────────────────────────────────────────────────────
with open(SIMPLIFIED_CHUNKS_FILE, "r", encoding="utf-8") as f:
    simplified_chunks = json.load(f)

with open(BOOK_STRUCTURE_FILE, "r", encoding="utf-8") as f:
    chapters_metadata = json.load(f)

print(f"📦 Loaded {len(simplified_chunks)} simplified chunks")
print(f"📚 Original structure: {len(chapters_metadata)} chapters")

In [ ]:
# ── Trim Overlaps ────────────────────────────────────────────────────────────
# Remove duplicate text at chunk boundaries caused by the overlap strategy.

trimmed_chunks = trim_overlaps(simplified_chunks, overlap_tokens=OVERLAP_TOKENS)
print(f"✂️  Trimmed overlaps from {len(trimmed_chunks)} chunks")

In [ ]:
# ── Assemble into Book ───────────────────────────────────────────────────────
# Stitch chunks back together with chapter headings and section structure.

book_markdown = assemble_book(trimmed_chunks, chapters_metadata)

# Add table of contents
toc = generate_toc(book_markdown)
final_book = toc + "\n\n---\n\n" + book_markdown

print(f"📖 Assembled book: {len(final_book):,} characters")

In [ ]:
# ── Quality Review: Length Comparison ────────────────────────────────────────
# Check that simplified text is roughly same length as original (90-120%).
# Much shorter = information is being dropped. Much longer = too verbose.

length_report = compare_lengths(
    original_chunks=simplified_chunks,  # has both "text" and "simplified_text"
    simplified_chunks=simplified_chunks,
)

print(f"📊 Length Comparison:")
print(f"   Original:   {length_report['original_words']:,} words")
print(f"   Simplified: {length_report['simplified_words']:,} words")
print(f"   Ratio:      {length_report['ratio']:.2f}x")

if length_report['flagged_chunks']:
    print(f"\n   ⚠️  {len(length_report['flagged_chunks'])} chunks have suspicious length ratios:")
    for cid in length_report['flagged_chunks']:
        print(f"      Chunk {cid}: ratio = {length_report['per_chunk_ratios'][cid]:.2f}")

In [ ]:
# ── Preview: Side-by-Side Comparison ─────────────────────────────────────────
# Show a random chunk — original vs simplified — so you can visually verify.

import random
sample = random.choice(simplified_chunks)

print("="*80)
print(f"SAMPLE — Chunk {sample['chunk_id']} (Chapter: {sample['chapter']})")
print("="*80)
print("\n🔴 ORIGINAL (first 500 chars):")
print(sample["text"][:500])
print("\n🟢 SIMPLIFIED (first 500 chars):")
print(sample["simplified_text"][:500])

In [ ]:
# ── Save Final Output ────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / "simplified_book.md"

saved_path = save_output(final_book, output_path)

print(f"\n🎉 DONE! Simplified book saved to:")
print(f"   📄 {saved_path}")
print(f"   📏 {len(final_book):,} characters")
print(f"\nOpen the file in any Markdown viewer/editor to read your simplified book!")